Hikyuu 交互式工具示例
==============

1、引入交互式工具
-----------------
需从hikyuu.interactive引入，而不是直接从hikyuu库中引入（hikyuu是一个库，可用于编制其他的工具，而hikyuu.interactive是基于hikyuu库实现的交互式探索工具）

In [ ]:
#%matplotlib inline
%time from hikyuu.interactive import *
#use_draw_engine('echarts') #use_draw_engine('matplotlib')  #the default is 'matplotlib' drawing

2、创建交易系统并运行
--------------------

In [ ]:
# Create a simulated trading account for backtesting with an initial capital of 300,000
my_tm = crtTM(init_cash = 300000)

# Create the signal generator (the 5-day EMA as the fast line and the 10-day EMA of the 5-day EMA itself as the slow line; buy when the fast line crosses the slow line upward, and sell otherwise)
my_sg = SG_Flex(EMA(C, n=5), slow_n=10)

# Fixedly buy 1000 shares each time
my_mm = MM_FixedCount(1000)

# Create the trading system and run it
sys = SYS_Simple(tm = my_tm, sg = my_sg, mm = my_mm)
sys.run(sm['sz000001'], Query(-150))

3、绘制曲线观察
---------------

In [ ]:
# Draw the system signals
sys.plot()

k = sm['sz000001'].get_kdata(Query(-150))
c = CLOSE(k)
fast = EMA(c, 5)
slow = EMA(fast, 10)

# Draw the signal generator using two indicators
fast.plot(new=False)
slow.plot(new=False)

4、绘制资金收益曲线
---------------------

In [ ]:
# Draw the equity return curve
x = my_tm.get_profit_curve(k.get_datetime_list(), Query.DAY)
x = PRICELIST(x)
x.plot()

5、回测统计报告
----------------------

In [ ]:
# Backtest statistics
from datetime import datetime

per = my_tm.get_performance()
print(per.to_df())


6、关于性能
---------------

经常有人问到性能问题，下面这段的代码使用之前的系统示例，遍历指定板块的所有股票，计算他们的“盈利交易比例%”（即胜率）。

In [ ]:
def test_func(stock, query):
    """Calculate the win rate of the system strategy for the given stock，the system strategy is the previous simple double-moving-average cross system (fixedly buying 100 shares each time)
    """
    # Create a simulated trading account for backtesting with an initial capital of 300,000
    my_tm = crtTM(init_cash = 1000000)

    # Create the signal generator (the 5-day EMA as the fast line and the 10-day EMA of the 5-day EMA itself as the slow line; buy when the fast line crosses the slow line upward, and sell otherwise)
    my_sg = SG_Flex(EMA(C, n=5), slow_n=10)

    # Fixedly buy 1000 shares each time
    my_mm = MM_FixedCount(100)

    # Create the trading system and run it
    sys = SYS_Simple(tm = my_tm, sg = my_sg, mm = my_mm)
    sys.run(stock, query)
    
    per = my_tm.get_performance(ext=False)
    return per["Winning Trade Ratio %"]

def total_func(blk, query):
    """Traverse all the stocks of the specified block and calculate the system win rate"""
    result = {}
    for s in blk:
        if s.valid and s.type != constant.STOCKTYPE_INDEX:
            result[s.name] = test_func(s, query)
    return result

遍历所有当前有效且并非指数的证券。下面是我的机器执行结果，共计算4151支证券，最近500个交易日，共耗时2.89秒。机器配置：Intel i7-4700HQ 2.G。

In [ ]:
# Wait until all the data is loaded before testing the subsequent actual calculation performance (it is not needed in actual use; otherwise the test time includes the first data loading time)
import time
while not sm.data_ready:
    time.sleep(0.5)

In [ ]:
%time a = total_func(blocka, Query(-500))
print(f"Number of securities calculated: {len(a)}")

In [ ]:
# 7000+ stocks over 10 years, 5~10x faster than vectorBT
%time a = total_func(blocka, Query(Datetime(20150101), Datetime(20260101)))
print(f"Number of securities calculated: {len(a)}")